In [ ]:
%cd ../../

In [ ]:
from datetime import time
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import seaborn as sns
import polars as pl
from sklearn.ensemble import IsolationForest

In [ ]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Read data

In [ ]:
path_root = Path("data/raw/occupancy_supersight")
paths = [path for path in path_root.glob("./**/*.xlsx")]

occu_raw = pl.concat([pl.read_excel(path) for path in paths])

occu_raw.head()

In [ ]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

raw = []
for path in paths:
    df = pl.read_csv(path, separator=';')
    df.columns = [str(x) for x in np.arange(df.shape[1])]
    raw.append(df)


pos_raw = pl.concat(raw)
pos_raw.head()

# Process

## POS

In [ ]:
map_resname = pl.DataFrame([
    {'restaurant_full': '610 Physicum', 'restaurant': 'physicum'},
    {'restaurant_full': '600 Chemicum', 'restaurant': 'chemicum'},
    {'restaurant_full': '570 Viikuna', 'restaurant': 'viikuna'},
    {'restaurant_full': '620 Exactum', 'restaurant': 'exactum'},
])

map_resname.head()

In [ ]:
pos_raw.columns = [
    'date',
    'time',
    'restaurant_full',
    'meal_type',
    'meal',
    'pcs',
    'co2',
]

pos = (
    pos_raw
    .with_columns(
        pl.concat_str(['date', 'time']).str.to_datetime("%d.%m.%Y %H:%M").alias('datetime')
        # pl.col('date').str.to_date("%d.%m.%Y"),
        # pl.col('time').str.to_time("%H:%M"),
    )

    .join(map_resname, on='restaurant_full')

    .select('datetime', 'restaurant', 'pcs')
)

pos.head()

## Occupancy

In [ ]:
path = "data/raw/occupancy_supersight/mapping_phoneName.csv"
map_phoneName2location = pl.read_csv(path)

map_phoneName2location

In [ ]:
occu = (
    occu_raw
    .with_columns(pl.col('dateCreated').str.to_datetime())
    .join(map_phoneName2location, on='phoneName', how='left')
    .drop('phoneName')
    .rename({'dateCreated': 'datetime'})
    
    .with_columns(
        pl.col('datetime').dt.convert_time_zone('Europe/Helsinki')
    )
)

occu.head()

In [ ]:
(
    occu
    .group_by('location')
    .agg(
        pl.col('datetime').min().alias('datetime_min'),
        pl.col('datetime').max().alias('datetime_max'),
    )
)

# EDA

### Prepare common dim tables

In [ ]:
def gen_timeslot(
    restaurant: Literal["chemicum", "viikuna", "physicum", "exactum"],
    resolution: int
):
    time_start, time_end = None, None
    match (restaurant):
        case "chemicum":
            time_start = time(10, 30)
            time_end = time(15)
        case "viikuna":
            time_start = time(10, 30)
            time_end = time(14)
        case "physicum":
            time_start = time(10)
            time_end = time(16, 30)
        case "exactum":
            time_start = time(11)
            time_end = time(14)
        case _:
            raise NotImplementedError()

    timeslot = (
        pl.DataFrame({
            'time_start': pl.time_range(time_start, time_end, f"{resolution}m", eager=True),
        })
        .with_columns(
            pl.col('time_start').shift(-1).alias('time_end')
        )
        .drop_nulls()
        .with_row_index('slot')
    )

    return timeslot

gen_timeslot('chemicum', 1).head()

In [ ]:
map_weekday2name = { 
    1: 'Monday',
    2: 'Tuesday',
    3: 'Wednesday',
    4: 'Thursday',
    5: 'Friday',
}

### Visualize total for specific location for `occupancy` and `pos`

In [ ]:
# location = 'chemicum_unicafe_entrance'
# restaurant = 'chemicum'
# resolution = 3



# # Determine start and end dates
# dates_pos = (
#     pos
#     .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
#     .filter(pl.col('restaurant') == pl.lit(restaurant))
#     .select(
#         pl.col('datetime').dt.date().min().alias('date_min'),
#         pl.col('datetime').dt.date().max().alias('date_max')
#     )
#     .to_dicts()
# )[0]

# dates_occu = (
#     occu
#     .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
#     .filter(pl.col('location') == pl.lit(location))
#     .select(
#         pl.col('datetime').dt.date().min().alias('date_min'),
#         pl.col('datetime').dt.date().max().alias('date_max')
#     )
#     .to_dicts()
# )[0]

# date_start = max(dates_occu['date_min'], dates_pos['date_min'])
# date_end = min(dates_occu['date_max'], dates_pos['date_max'])



# # Extract data
# timeslot = gen_timeslot(restaurant, resolution)
# pos_res = (
#     pos
#     .filter(
#         (pl.col('datetime').dt.weekday() < 6)           # Filter out Saturday and Sunday

#         & (pl.col('restaurant') == pl.lit(restaurant))  # Keep specified restaurant

#         & (pl.col('datetime').dt.date() >= date_start)
#         & (pl.col('datetime').dt.date() <= date_end)
#     )

#     .join(timeslot, how='cross')
#     .filter(
#         (pl.col('datetime').dt.time() >= pl.col('time_start'))
#         & (pl.col('datetime').dt.time() < pl.col('time_end'))
#     )
#     .drop('time_start', 'time_end')
#     .group_by('slot')
#     .agg(
#         pl.col('pcs').sum(),
#     )
# )

# occu_res = (
#     occu
#     .filter(
#         (pl.col('datetime').dt.weekday() < 6)           # Filter out Saturday and Sunday

#         & (pl.col('location') == pl.lit(location))      # Keep specified restaurant

#         & (pl.col('datetime').dt.date() >= date_start)
#         & (pl.col('datetime').dt.date() <= date_end)
#     )

#     .join(timeslot, how='cross')
#     .filter(
#         (pl.col('datetime').dt.time() >= pl.col('time_start'))
#         & (pl.col('datetime').dt.time() < pl.col('time_end'))
#     )
#     .drop('time_start', 'time_end')

#     .group_by('slot')
#     .agg(
#         pl.col('countIn').sum(),
#         pl.col('countOut').sum(),
#     )
# )

# df = (
#     occu_res
#     .join(pos_res, on='slot')

#     .join(timeslot, on='slot')

#     .with_columns(
#         pl.col('countIn').cum_sum().over(pl.lit(1), order_by='time_start').alias('in_cum'),
#         pl.col('countOut').cum_sum().over(pl.lit(1), order_by='time_start').alias('out_cum')
#     )
#     # .with_columns(
#     #     (pl.col('in_cum') - pl.col('out_cum')).alias('occupancy')
#     # )

#     # Unpivot
#     .unpivot(index=['slot', 'time_start'], on=['countIn', 'countOut', 'pcs'], value_name='count', variable_name='type')


#     .to_pandas()
# )




# # Plot
# df['time_start'] = pd.to_datetime(df['time_start'], format="%H:%M:%S")

# fig = plt.figure(figsize=(8, 6))
# ax = fig.add_subplot(111)

# sns.lineplot(df, y='count', x='time_start', ax=ax, hue='type')
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlabel("Time")
# ax.set_title(f"Total in {restaurant} | resolution = {resolution} minutes", fontdict={'weight': 'bold', 'size': 14})
# ax.tick_params(axis='x', labelrotation = 45)

### Visualize total per restaurant on specific date

In [ ]:
location = 'chemicum_unicafe_entrance'
restaurant = 'chemicum'
date_plot = '2024-10-30'
resolution = 1



# Prepare data
timeslot = gen_timeslot(restaurant, resolution)


occu_res = (
    occu
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
    
    .filter(
        (pl.col('location') == pl.lit(location))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )
    .join(timeslot, how='cross')
    .filter(
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')

    .group_by('slot')
    .agg(
        pl.col('countIn').sum(),
        pl.col('countOut').sum(),
    )
)


pos_res = (
    pos
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday

    .filter(
        (pl.col('restaurant') == pl.lit(restaurant))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )

    .join(timeslot, how='cross')
    .filter(
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')
    .group_by('slot')
    .agg(
        pl.col('pcs').count(),      # This uses `count` instead of `sum` since it count no. orders, and this may differ from no. pcs
    )
)


df = (
    occu_res
    .join(pos_res, on='slot')

    .join(timeslot, on='slot')

    .with_columns(
        pl.col('countIn').cum_sum().over(pl.lit(1), order_by='time_start').alias('in_cum'),
        pl.col('countOut').cum_sum().over(pl.lit(1), order_by='time_start').alias('out_cum')
    )
    .with_columns(
        (pl.col('in_cum') - pl.col('out_cum')).alias('occupancy'),
        (pl.col('countIn') - pl.col('pcs')).alias('count_diff'),
    )

    # Unpivot
    .unpivot(index=['slot', 'time_start'], on=[
        'countIn', 
        # 'countOut', 
        'pcs', 
        # 'occupancy'
        # 'count_diff'
    ], value_name='count', variable_name='type')


    .to_pandas()
)



# Plot
df['time_start'] = pd.to_datetime(df['time_start'], format="%H:%M:%S")


fig = plt.figure(figsize=(12, 6))
fig.suptitle(f"{restaurant} on {date_plot} | resolution = {resolution} minutes", fontweight='bold', fontsize=16)

ax = fig.add_subplot(111)
sns.lineplot(df[df['type'] != 'count_diff'], y='count', x='time_start', ax=ax, hue='type')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel("Time")
ax.set_title('countIn, countOut and pcs')
ax.tick_params(axis='x', labelrotation = 45)

# ax = fig.add_subplot(122)
# sns.lineplot(df[df['type'] == 'count_diff'], y='count', x='time_start', ax=ax, hue='type')
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlabel("Time")
# ax.set_title('Solution 1: Diff between countIn and pcs')
# ax.axhline(y = 5, color = 'r', linestyle = '--')
# ax.tick_params(axis='x', labelrotation = 45)

### Visualize total per restaurant and weekday

In [ ]:
# restaurant = 'physicum'

# # Determine start and end dates
# dates_pos = (
#     pos
#     .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
#     .filter(pl.col('restaurant') == pl.lit(restaurant))
#     .select(
#         pl.col('datetime').dt.date().min().alias('date_min'),
#         pl.col('datetime').dt.date().max().alias('date_max')
#     )
#     .to_dicts()
# )[0]

# dates_occu = (
#     occu
#     .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
#     .filter(pl.col('restaurant') == pl.lit(restaurant))
#     .select(
#         pl.col('datetime').dt.date().min().alias('date_min'),
#         pl.col('datetime').dt.date().max().alias('date_max')
#     )
#     .to_dicts()
# )[0]

# date_start = max(dates_occu['date_min'], dates_pos['date_min'])
# date_end = min(dates_occu['date_max'], dates_pos['date_max'])


# occu_res = (
#     occu
#     .filter(
#         (pl.col('datetime').dt.weekday() < 6)           # Filter out Saturday and Sunday

#         & (pl.col('restaurant') == pl.lit(restaurant))  # Keep specified restaurant

#         & (pl.col('datetime').dt.date() >= date_start)
#         & (pl.col('datetime').dt.date() <= date_end)
#     )
#     .join(timeslot, how='cross')
#     .filter(
#         (pl.col('datetime').dt.time() >= pl.col('time_start'))
#         & (pl.col('datetime').dt.time() < pl.col('time_end'))
#     )
#     .drop('time_start', 'time_end')

#     .with_columns(
#         pl.col('datetime').dt.weekday().cast(pl.Int64).alias('weekday')
#     )

#     .group_by('weekday', 'slot')
#     .agg(
#         pl.col('countIn').sum(),
#         pl.col('countOut').sum(),
#     )
# )


# pos_res = (
#     pos
#     .filter(
#         (pl.col('datetime').dt.weekday() < 6)           # Filter out Saturday and Sunday

#         & (pl.col('restaurant') == pl.lit(restaurant))  # Keep specified restaurant

#         & (pl.col('datetime').dt.date() >= date_start)
#         & (pl.col('datetime').dt.date() <= date_end)
#     )

#     .join(timeslot, how='cross')
#     .filter(
#         (pl.col('datetime').dt.time() >= pl.col('time_start'))
#         & (pl.col('datetime').dt.time() < pl.col('time_end'))
#     )
#     .drop('time_start', 'time_end')

#     .with_columns(
#         pl.col('datetime').dt.weekday().cast(pl.Int64).alias('weekday')
#     )

#     .group_by('weekday', 'slot')
#     .agg(
#         pl.col('pcs').sum(),
#     )
# )


# df = (
#     occu_res
#     .join(pos_res, on=['weekday', 'slot'])

#     .join(timeslot, on='slot')

#     # Unpivot
#     .unpivot(index=['weekday', 'slot', 'time_start'], on=['countIn', 'countOut', 'pcs'], value_name='count', variable_name='type')

#     # .join(map_weekday2name, on='weekday')
#     # .drop('weekday')
#     # .rename({'name': 'weekday'})


#     .to_pandas()
# )





# # Plot
# df['time_start'] = pd.to_datetime(df['time_start'], format="%H:%M:%S")
# weekdays = sorted(df['weekday'].unique())

# fig = plt.figure(figsize=(12, 10))
# fig.subplots_adjust(wspace=0.2, hspace=0.6, top=0.9)
# fig.suptitle(f"Total in {restaurant} per weekday", fontweight='bold', fontsize=16)

# for idx, weekday in enumerate(weekdays):
#     ax = fig.add_subplot(3, 2, idx + 1)

#     sns.lineplot(df[df['weekday'] == weekday], y='count', x='time_start', ax=ax, hue='type')
#     ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
#     ax.set_xlabel("Time")
#     ax.set_title(map_weekday2name[weekday])
#     ax.tick_params(axis='x', labelrotation = 45)

# Queue detection

In [ ]:
location = 'chemicum_unicafe_entrance'
restaurant = 'chemicum'
date_plot = '2024-10-30'
resolution = 1

### Solution 1: Measuring the absolute difference

In [ ]:
# location = 'chemicum_unicafe_entrance'
# restaurant = 'chemicum'
# date_plot = '2024-10-30'
# resolution = 3



# Prepare data
timeslot = gen_timeslot(restaurant, resolution)


occu_res = (
    occu
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
    
    .filter(
        (pl.col('location') == pl.lit(location))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )
    .join_where(
        timeslot,
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')

    .group_by('slot')
    .agg(
        pl.col('countIn').sum(),
        pl.col('countOut').sum(),
    )
)


pos_res = (
    pos
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday

    .filter(
        (pl.col('restaurant') == pl.lit(restaurant))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )

    .join_where(
        timeslot,
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')
    .group_by('slot')
    .agg(
        pl.col('pcs').count(),
    )
)


df = (
    occu_res
    .join(pos_res, on='slot')

    .join(timeslot, on='slot')

    .with_columns(
        pl.col('countIn').cum_sum().over(pl.lit(1), order_by='time_start').alias('in_cum'),
        pl.col('countOut').cum_sum().over(pl.lit(1), order_by='time_start').alias('out_cum')
    )
    .with_columns(
        (pl.col('in_cum') - pl.col('out_cum')).alias('occupancy'),
        (pl.col('countIn') - pl.col('pcs')).alias('count_diff'),
    )

    # Unpivot
    .unpivot(index=['slot', 'time_start'], on=[
        'countIn', 
        # 'countOut', 
        'pcs', 
        # 'occupancy'
        'count_diff'
    ], value_name='count', variable_name='type')


    .to_pandas()
)


# Determine the anomaly
occu_queue = df[df['type'] == 'count_diff'][['time_start', 'count']].copy()
occu_queue['anomaly'] = IsolationForest(contamination=0.05).fit_predict(occu_queue['count'].to_numpy()[:, None])
occu_queue = occu_queue[(occu_queue['anomaly'] == -1) & (occu_queue['count'] > 0)]
occu_queue['time_start'] = pd.to_datetime(occu_queue['time_start'], format="%H:%M:%S")


# Plot
df['time_start'] = pd.to_datetime(df['time_start'], format="%H:%M:%S")


fig = plt.figure(figsize=(8, 6))
fig.suptitle(f"Solution 1 | {restaurant} on {date_plot} | resolution = {resolution} minutes", fontweight='bold', fontsize=16)

# ax = fig.add_subplot(121)
# sns.lineplot(df[df['type'] != 'count_diff'], y='count', x='time_start', ax=ax, hue='type')
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlabel("Time")
# ax.set_title('countIn and pcs')
# ax.tick_params(axis='x', labelrotation = 45)

ax = fig.add_subplot(111)
sns.lineplot(df[df['type'] == 'count_diff'], y='count', x='time_start', ax=ax, hue='type')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel("Time")
ax.set_title('Diff between countIn and pcs')
ax.tick_params(axis='x', labelrotation = 45)


sns.scatterplot(occu_queue, x='time_start', y='count', markers='o', s=70, c='r')

### Solution 2: Applying anomaly detection with standard deviation

In [ ]:
# Generate the timeslot for all restaurants
dfs = []
for res in ["chemicum", "viikuna", "physicum", "exactum"]:
    df = gen_timeslot(res, resolution)
    df = df.with_columns(pl.lit(res).alias('restaurant'))
    dfs.append(df)

timeslots = pl.concat(dfs)



# Calcualte the mean and std of pcs per restaurant and slot
pos_info = (
    pos
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday

    .join_where(
        timeslots,
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
        & (pl.col('restaurant') == pl.col('restaurant_right'))
    )
    .with_columns(
        pl.col('datetime').dt.date().alias('date')
    )
    .group_by('restaurant', 'slot', 'date')
    .agg(
        pl.col('pcs').sum()
    )
    .group_by('restaurant', 'slot')
    .agg(
        pl.col('pcs').mean().alias('pcs_mean'),
        pl.col('pcs').std().alias('pcs_std'),
    )
)

pos_info.head()

In [ ]:
THETA = 3

# Prepare data
timeslot = gen_timeslot(restaurant, resolution)


occu_queue = (
    occu
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
    
    .filter(
        (pl.col('location') == pl.lit(location))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )
    .join_where(
        timeslot,
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')

    .group_by('slot')
    .agg(
        pl.col('countIn').sum(),
    )


    # Get mean and std pos
    .join(
        pos_info.filter(pl.col('restaurant') == pl.lit(restaurant)),
        on='slot',
        how='left'
    )
    .filter(pl.col('countIn') >= pl.col('pcs_mean') * THETA)


    # # Supplement timing info for slots
    .join(timeslot, on='slot')


    # Remove redundant columns
    .drop('restaurant', 'pcs_mean', 'pcs_std', 'slot', 'time_end')


    .to_pandas()
)

occu_queue['time_start'] = pd.to_datetime(occu_queue['time_start'], format="%H:%M:%S")

occu_queue

In [ ]:
# Prepare data
timeslot = gen_timeslot(restaurant, resolution)


occu_res = (
    occu
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
    
    .filter(
        (pl.col('location') == pl.lit(location))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )
    .join_where(
        timeslot,
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')

    .group_by('slot')
    .agg(
        pl.col('countIn').sum(),
        pl.col('countOut').sum(),
    )
)

pos_res = (
    pos
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday

    .filter(
        (pl.col('restaurant') == pl.lit(restaurant))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )

    .join_where(
        timeslot,
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')
    .group_by('slot')
    .agg(
        pl.col('pcs').count(),
    )
)


df = (
    occu_res
    .join(pos_res, on='slot')

    .join(timeslot, on='slot')

    .with_columns(
        pl.col('countIn').cum_sum().over(pl.lit(1), order_by='time_start').alias('in_cum'),
        pl.col('countOut').cum_sum().over(pl.lit(1), order_by='time_start').alias('out_cum')
    )
    .with_columns(
        # (pl.col('in_cum') - pl.col('out_cum')).alias('occupancy'),
        (pl.col('countIn') - pl.col('pcs')).alias('count_diff'),
    )

    # Unpivot
    .unpivot(index=['slot', 'time_start'], on=[
        'countIn', 
        # 'countOut', 
        'pcs', 
        # 'occupancy'
        # 'count_diff'
    ], value_name='count', variable_name='type')


    .to_pandas()
)



# Plot
df['time_start'] = pd.to_datetime(df['time_start'], format="%H:%M:%S")


fig = plt.figure(figsize=(8, 6))
fig.suptitle(f"Solution 2 | {restaurant} on {date_plot} | resolution = {resolution} minutes", fontweight='bold', fontsize=16)

ax = fig.add_subplot(111)
sns.lineplot(df[df['type'] != 'count_diff'], y='count', x='time_start', ax=ax, hue='type')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel("Time")
ax.set_title('countIn and pcs')
ax.tick_params(axis='x', labelrotation = 45)


sns.scatterplot(occu_queue, x='time_start', y='countIn', markers='o', s=70, c='r')

# Queue length estimation

In [ ]:
location = 'chemicum_unicafe_entrance'
restaurant = 'chemicum'
resolution = 1

In [ ]:
timeslot = gen_timeslot(restaurant, resolution)

## Find maximum serving rate

In [ ]:
# start_peak = pl.lit('11:45').str.to_time("%H:%M")
# end_peak = pl.lit('12:15').str.to_time("%H:%M")

(
    pos
    .filter(
        (1 == 1)

        # Filter out Saturday and Sunday
        & (pl.col('datetime').dt.weekday() < 6)
        
        # Filter records in specified restaurant
        & (pl.col('restaurant') == pl.lit(restaurant))      
        # & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())

        # Filter records in peak time
        # & (pl.col('datetime').dt.time() >= start_peak)
        # & (pl.col('datetime').dt.time() <= end_peak)
    )

    .join(timeslot, how='cross')
    .filter(
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')


    .with_columns(pl.col('datetime').dt.date().alias('date'))
    .group_by('date', 'slot')
    .len('num_orders')
    .group_by('slot')
    .agg(pl.col('num_orders').median())

    .join(timeslot, on='slot', how='right')
    .sort('slot')

    # .write_excel(f"mu_{restaurant}_by_slot.xlsx")
).head()

## Find queue length

In [ ]:
location = 'chemicum_unicafe_entrance'
restaurant = 'chemicum'
date_plot = '2024-10-09'
resolution = 1
SERVING_RATE = 4.9

In [ ]:
# df = (
#     occu
#     .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
#
#     .filter(
#         (pl.col('location') == pl.lit(location))
#         & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
#     )
#     .join(timeslot, how='cross')
#     .filter(
#         (pl.col('datetime').dt.time() >= pl.col('time_start'))
#         & (pl.col('datetime').dt.time() < pl.col('time_end'))
#     )
#     .drop('time_start', 'time_end')
#
#     .group_by('slot')
#     .agg(
#         pl.col('countIn').sum(),
#     )
#
#     .join(timeslot, on='slot', how='right')
#     .sort('slot')
#
#     # Calculate queue length
#     # .with_row_index('ith', offset=1)
#     .select(
#         'slot', 'time_start', 'time_end',
#         pl.col('countIn').fill_null(0),
#         pl.col('countIn').fill_null(0).cum_sum().alias('cumCount'),
#         # 'ith',
#     )
#
#     # .with_columns(
#     #     pl.max_horizontal(
#     #         pl.col('cumCount') - pl.col('ith') * SERVING_RATE,
#     #         pl.lit(0)
#     #     ).alias('queue_size')
#     #     # ().alias('remaining')
#     # )
#     # .select('slot', 'queue_size')
#
#     # .write_excel('chemicum_unicafe_entrance_2024-10-30.xlsx')
# )
#
# df.head()

In [ ]:
# remainings = []
#
# for idx, countIn in enumerate(df['countIn']):
#     remaining = 0 if idx == 0 else remainings[idx - 1]
#
#     remaining = float(max(0, countIn + remaining - SERVING_RATE))
#     remainings.append(remaining)
#
# df = df.with_columns(pl.Series(remainings).alias('queue_size'))
# df.head()

In [ ]:
queue_size = (
    occu
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday

    .filter(
        (pl.col('location') == pl.lit(location))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )
    .join(timeslot, how='cross')
    .filter(
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')

    .group_by('slot')
    .agg(
        pl.col('countIn').sum(),
    )

    .join(timeslot, on='slot', how='right')
    .sort('slot')

    # Calculate queue length
    # .with_row_index('ith', offset=1)
    .select(
        'slot', 'time_start', 'time_end',
        pl.col('countIn').fill_null(0),
        pl.col('countIn').fill_null(0).cum_sum().alias('cumCount'),
        # 'ith',
    )
    # .with_columns(
    #     pl.max_horizontal(
    #         pl.col('cumCount') - pl.col('ith') * SERVING_RATE,
    #         pl.lit(0)
    #     ).alias('queue_size')
    #     # ().alias('remaining')
    # )
    # .select('slot', 'queue_size')

    # .write_excel('chemicum_unicafe_entrance_2024-10-30.xlsx')
)


# Add queue size
remainings = []

for idx, countIn in enumerate(queue_size['countIn']):
    remaining = 0 if idx == 0 else remainings[idx - 1]

    remaining = float(max(0, countIn + remaining - SERVING_RATE))
    remainings.append(remaining)

queue_size = queue_size.with_columns(pl.Series(remainings).alias('queue_size'))





# Prepare data
timeslot = gen_timeslot(restaurant, resolution)


occu_res = (
    occu
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday
    
    .filter(
        (pl.col('location') == pl.lit(location))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )
    .join(timeslot, how='cross')
    .filter(
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')

    .group_by('slot')
    .agg(
        pl.col('countIn').sum(),
        pl.col('countOut').sum(),
    )
)


pos_res = (
    pos
    .filter(pl.col('datetime').dt.weekday() < 6)        # Filter out Saturday and Sunday

    .filter(
        (pl.col('restaurant') == pl.lit(restaurant))
        & (pl.col('datetime').dt.date() == pl.lit(date_plot).str.to_date())
    )

    .join(timeslot, how='cross')
    .filter(
        (pl.col('datetime').dt.time() >= pl.col('time_start'))
        & (pl.col('datetime').dt.time() < pl.col('time_end'))
    )
    .drop('time_start', 'time_end')
    .group_by('slot')
    .agg(
        pl.col('pcs').count(),      # This uses `count` instead of `sum` since it count no. orders, and this may differ from no. pcs
    )
)


df = (
    occu_res
    .join(pos_res, on='slot')

    .join(timeslot, on='slot')

    .join(queue_size, on='slot')

    # Unpivot
    .unpivot(index=['slot', 'time_start'], on=[
        'countIn', 
        # 'pcs', 
        'queue_size', 
    ], value_name='count', variable_name='type')


    .to_pandas()
)



# Plot
df['time_start'] = pd.to_datetime(df['time_start'], format="%H:%M:%S")


fig = plt.figure(figsize=(12, 6))
fig.suptitle(f"{restaurant} on {date_plot} | resolution = {resolution} minutes | serving_rate = {SERVING_RATE} persons/min", fontweight='bold', fontsize=16)

ax = fig.add_subplot(111)
sns.lineplot(df[df['type'] != 'count_diff'], y='count', x='time_start', ax=ax, hue='type')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel("Time")
ax.axhline(y = 35, color = 'r', linestyle = '--')
ax.tick_params(axis='x', labelrotation = 45)